# 04 — Séries Temporais: Média Móvel e Análise de Tendência

Este notebook aplica técnicas de séries temporais:
- **Média móvel** (3 e 6 meses) para suavização
- **Bandas de desvio-padrão** para faixas de variação
- **Decomposição** da série em tendência, sazonalidade e resíduo
- **Análise orçamento vs. realizado** cruzando as 3 fontes do ETL

Todas as visualizações são salvas em `outputs/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
import os

os.makedirs("outputs", exist_ok=True)

# Estilo consistente
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#fafafa",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10
})

print("Bibliotecas carregadas.")

## 4.1 — Carregar dados processados

In [ ]:
df = pd.read_parquet("data/processed/finance_clean.parquet")
df_orc = pd.read_parquet("data/processed/orcamento_clean.parquet")
df_ibov = pd.read_parquet("data/processed/ibovespa_clean.parquet")

print(f"Transações: {len(df)} registros")
print(f"Orçamento:  {len(df_orc)} registros")
print(f"Ibovespa:   {len(df_ibov)} registros")

## 4.2 — Série temporal: receita mensal

In [ ]:
# Separar receita e despesa por mês
receita_mensal = (
    df[df["type"] == "receita"]
    .set_index("date")
    .resample("ME")["amount"]
    .sum()
)

despesa_mensal = (
    df[df["type"] == "despesa"]
    .set_index("date")
    .resample("ME")["amount"]
    .sum()
)

saldo_mensal = receita_mensal.subtract(despesa_mensal, fill_value=0)

print(f"Período: {receita_mensal.index.min().strftime('%Y-%m')} a {receita_mensal.index.max().strftime('%Y-%m')}")
print(f"Meses com dados: {len(receita_mensal)}")

## 4.3 — Média Móvel + Bandas de Desvio-Padrão

In [ ]:
# Médias móveis
mm3 = receita_mensal.rolling(3).mean()
mm6 = receita_mensal.rolling(6).mean()

# Banda de desvio-padrão (rolling 3 meses)
std3 = receita_mensal.rolling(3).std()

print("Média móvel calculada (janelas de 3 e 6 meses).")

In [ ]:
# ── GRÁFICO 1: Série temporal com média móvel ──
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(receita_mensal.index, receita_mensal.values,
        "o-", color="#94a3b8", markersize=4, linewidth=1, alpha=0.6, label="Receita Mensal")

ax.plot(mm3.index, mm3.values,
        color="#3b82f6", linewidth=2.5, label="Média Móvel 3M")

ax.plot(mm6.index, mm6.values,
        color="#8b5cf6", linewidth=2, linestyle="--", label="Média Móvel 6M")

# Banda ±1σ
ax.fill_between(mm3.index,
                (mm3 - std3).values,
                (mm3 + std3).values,
                alpha=0.12, color="#3b82f6", label="Banda ±1σ (3M)")

ax.set_title("Série Temporal — Receita Mensal com Média Móvel", fontsize=13, fontweight="bold")
ax.set_xlabel("Mês")
ax.set_ylabel("Receita (R$)")
ax.legend(loc="upper left", fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig("outputs/serie_temporal.png", dpi=150, bbox_inches="tight")
plt.show()
print("✔ Gráfico salvo em outputs/serie_temporal.png")

## 4.4 — Receita vs. Despesa ao longo do tempo

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Receita vs Despesa
ax1.bar(receita_mensal.index, receita_mensal.values, width=20,
        color="#22c55e", alpha=0.7, label="Receita")
ax1.bar(despesa_mensal.index, -despesa_mensal.values, width=20,
        color="#ef4444", alpha=0.7, label="Despesa")
ax1.axhline(0, color="black", linewidth=0.5)
ax1.set_title("Receita vs. Despesa Mensal", fontsize=12, fontweight="bold")
ax1.set_ylabel("Valor (R$)")
ax1.legend()

# Saldo acumulado
saldo_acum = saldo_mensal.cumsum()
colors = ["#22c55e" if v >= 0 else "#ef4444" for v in saldo_acum.values]
ax2.fill_between(saldo_acum.index, 0, saldo_acum.values,
                 where=saldo_acum.values >= 0, color="#22c55e", alpha=0.3)
ax2.fill_between(saldo_acum.index, 0, saldo_acum.values,
                 where=saldo_acum.values < 0, color="#ef4444", alpha=0.3)
ax2.plot(saldo_acum.index, saldo_acum.values, color="#1e40af", linewidth=2)
ax2.axhline(0, color="black", linewidth=0.5)
ax2.set_title("Saldo Acumulado", fontsize=12, fontweight="bold")
ax2.set_ylabel("Saldo Acumulado (R$)")
ax2.set_xlabel("Mês")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig("outputs/receita_despesa.png", dpi=150, bbox_inches="tight")
plt.show()
print("✔ Gráfico salvo em outputs/receita_despesa.png")

## 4.5 — Orçamento Planejado vs. Realizado

In [ ]:
# Orçamento: agrupar receita por mês
orc_mensal = df_orc.groupby("mes").agg({
    "receita": "first",  # receita é igual para todas as categorias no mesmo mês
    "despesa": "sum"
}).reset_index()
orc_mensal = orc_mensal.rename(columns={"mes": "date", "receita": "receita_orcada", "despesa": "despesa_orcada"})

# Cruzar com realizado
realizado = pd.DataFrame({
    "date": receita_mensal.index,
    "receita_real": receita_mensal.values,
    "despesa_real": despesa_mensal.reindex(receita_mensal.index, fill_value=0).values
})

comp = realizado.merge(orc_mensal, on="date", how="inner")
comp["desvio_receita_%"] = ((comp["receita_real"] - comp["receita_orcada"]) / comp["receita_orcada"] * 100).round(1)

print(f"Meses comparáveis: {len(comp)}")
comp.head(10)

In [ ]:
# ── GRÁFICO: Orçado vs Realizado ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Receita
ax1.plot(comp["date"], comp["receita_orcada"], "s--", color="#6b7280",
         markersize=4, label="Receita Orçada", linewidth=1.5)
ax1.plot(comp["date"], comp["receita_real"], "o-", color="#22c55e",
         markersize=4, label="Receita Real", linewidth=1.5)
ax1.set_title("Receita: Orçado vs. Realizado", fontsize=12, fontweight="bold")
ax1.set_ylabel("R$")
ax1.legend()

# Desvio %
colors = ["#22c55e" if v >= 0 else "#ef4444" for v in comp["desvio_receita_%"]]
ax2.bar(comp["date"], comp["desvio_receita_%"], width=20, color=colors, alpha=0.7)
ax2.axhline(0, color="black", linewidth=0.5)
ax2.set_title("Desvio % (Realizado − Orçado)", fontsize=12, fontweight="bold")
ax2.set_ylabel("Desvio (%)")
ax2.set_xlabel("Mês")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig("outputs/orcado_vs_realizado.png", dpi=150, bbox_inches="tight")
plt.show()
print("✔ Gráfico salvo em outputs/orcado_vs_realizado.png")

## 4.6 — Análise de Sazonalidade

In [ ]:
# Receita média por mês do ano
sazonalidade = receita_mensal.groupby(receita_mensal.index.month).agg(["mean", "std"])
sazonalidade.index.name = "mes_ano"
meses_pt = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(1, 13), sazonalidade["mean"], color="#3b82f6", alpha=0.7,
       yerr=sazonalidade["std"], capsize=4, error_kw={"elinewidth": 1.5})
ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses_pt)
ax.set_title("Sazonalidade — Receita Média por Mês", fontsize=13, fontweight="bold")
ax.set_ylabel("Receita Média (R$)")
ax.set_xlabel("Mês")

plt.tight_layout()
plt.savefig("outputs/sazonalidade.png", dpi=150, bbox_inches="tight")
plt.show()
print("✔ Gráfico salvo em outputs/sazonalidade.png")

## 4.7 — Dashboard resumo (4 gráficos em 1)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Pipeline ETL — Dashboard de Análise Financeira", fontsize=15, fontweight="bold", y=1.02)

# (0,0) Série temporal com MM
axes[0, 0].plot(receita_mensal.index, receita_mensal.values, "o-", color="#94a3b8", markersize=3, alpha=0.5)
axes[0, 0].plot(mm3.index, mm3.values, color="#3b82f6", linewidth=2)
axes[0, 0].fill_between(mm3.index, (mm3 - std3).values, (mm3 + std3).values, alpha=0.1, color="#3b82f6")
axes[0, 0].set_title("Receita Mensal + MM-3")
axes[0, 0].set_ylabel("R$")

# (0,1) Distribuição por tipo
tipo_total = df.groupby("type")["amount"].sum()
axes[0, 1].pie(tipo_total.abs().values, labels=["Despesa", "Receita"] if tipo_total.index[0] == "despesa" else ["Receita", "Despesa"],
               colors=["#ef4444", "#22c55e"], autopct="%1.1f%%", startangle=90)
axes[0, 1].set_title("Distribuição por Tipo")

# (1,0) Top categorias de despesa
top_desp = df[df["type"] == "despesa"].groupby("category")["amount"].sum().sort_values(ascending=True).tail(8)
axes[1, 0].barh(top_desp.index, top_desp.values, color="#f97316", alpha=0.8)
axes[1, 0].set_title("Top 8 Categorias de Despesa")
axes[1, 0].set_xlabel("Valor Total (R$)")

# (1,1) Ibovespa com MM
ibov_close = df_ibov.set_index("date")["Close"]
ibov_mm30 = ibov_close.rolling(30).mean()
axes[1, 1].plot(ibov_close.index, ibov_close.values, color="#94a3b8", alpha=0.4, linewidth=0.5)
axes[1, 1].plot(ibov_mm30.index, ibov_mm30.values, color="#8b5cf6", linewidth=2)
axes[1, 1].set_title("Ibovespa — Fechamento + MM-30")
axes[1, 1].set_ylabel("Pontos")

for ax in axes.flat:
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("outputs/dashboard_resumo.png", dpi=150, bbox_inches="tight")
plt.show()
print("✔ Dashboard salvo em outputs/dashboard_resumo.png")

---
## Resumo — Séries Temporais

| Análise | Técnica | Output |
|---------|---------|--------|
| Tendência | Média Móvel 3M e 6M | `outputs/serie_temporal.png` |
| Fluxo de caixa | Receita vs. Despesa + Saldo acumulado | `outputs/receita_despesa.png` |
| Planejamento | Orçado vs. Realizado (cruzamento de fontes) | `outputs/orcado_vs_realizado.png` |
| Sazonalidade | Receita média por mês do ano | `outputs/sazonalidade.png` |
| Visão geral | Dashboard com 4 painéis | `outputs/dashboard_resumo.png` |

---

### Projeto completo
Os 4 notebooks cobrem o pipeline ETL de ponta a ponta:
1. **Extração** — 3 fontes heterogêneas → Parquet
2. **Transformação** — Limpeza, tipagem, detecção de anomalias (Z-score 3σ)
3. **Regressão** — Projeção de receita 90 dias com validação R²
4. **Séries Temporais** — Média móvel, sazonalidade, orçado vs. real